# Gridworld problem demonstration

Suppose you are standing on one square of some $n$ by $n$ grid, and every time step you make a decision of whether to move up (U), down (D), left (L) or right (R), sometimes however there is a risk you end up somewhere you had not intended. If you are standing on the edge of the grid, you can not move off the edge of the grid. Every time you move on this grid, you gain or potentially lose a reward conditional on where you started. One corner of this grid, is the terminal node, once you enter it, you cannot leave it. Suppose you start randomly on this grid, we can use Value Iteration to tell us the best decision to move in on every square of the grid.

We give an example of this problem for when $n=2$, in which case we have 4 states which we denote TL (TopLeft), TR (TopRight), BL (BottomLeft), and finally the terminal node BR (BottomRight). The following table characterises the transition probabilities given a state-action pair. 

| State ($s$) | Action ($a$) | Next State ($s'$) | Probability $P(s'\mid s,a)$ |
|-------------|--------------|-------------------|-------------------------|
| TL        | R          | TR         | 0.9        |
| TL        | R          | BL         | 0.1        |
| TL        | D          | BL         | 0.9        |
| TL        | D          | TR         | 0.1        |
| TR        | L          | TL         | 0.9        |
| TR        | L          | BR         | 0.1        |
| TR        | D          | BR         | 0.8        |
| TR        | D          | TL         | 0.2        |
| BL        | R          | BR         | 0.9        |
| BL        | R          | TL         | 0.1        |
| BL        | U          | TL         | 0.8        |
| BL        | U          | BR         | 0.2        |
| BR        | U          | BR         | 1.0        |
| BR        | D          | BR         | 1.0        |
| BR        | R          | BR         | 1.0        |
| BR        | L          | BR         | 1.0        |

In code, we will represent this table as a dictionary whose keys are the state-action pairs, and values being a corresponding vector of transition probabilities:

In [1]:
S = ['TL', 'TR', 'BL', 'BR']  # STATE SPACE
A = ['R', 'L', 'D', 'U']     # ACTION SPACE

P = { #TRANSITION PROBABILITIES
    ('TL', 'R'): [0, 0.9, 0.1, 0],
    ('TL', 'D'): [0, 0.1, 0.9, 0],
    ('TR', 'L'): [0.9, 0, 0, 0.1],
    ('TR', 'D'): [0.2, 0, 0, 0.8],
    ('BL', 'R'): [0.1, 0, 0, 0.9],
    ('BL', 'U'): [0.8, 0, 0, 0.2]
} #State BR is terminal, so there is no associated probability vector

We can also characterise the rewards using the following table.

| State ($s$) | Action ($a$) | Next State ($s'$) | Reward $R(s,a,s')$ |
|-----------|------------|----------------|--------|
| TL        | R          | TR             | -1.0   |
| TL        | R          | BL             | -2.0   |
| TL        | D          | BL             | -2.0   |
| TL        | D          | TR             | -1.0   |
| TR        | L          | TL             | -1.5   |
| TR        | L          | BR             | 10.0   |
| TR        | D          | BR             | 15.0   |
| TR        | D          | TL             | -1.0   |
| BL        | R          | BR             | 20.0   |
| BL        | R          | TL             | -2.5   |
| BL        | U          | TL             | -0.5   |
| BL        | U          | BR             | 5.0    |

However, we observe that the value iteration algorithm does not need information of reward given the next state, it only needs the average reward given the state it is currently in and the action it takes. This relationship is characterised as follows:

$$
R(s,a) = \sum_{s'} R(s,a,s')P(s'|s,a)
$$

We therefore calculate the reward function $R(s,a)$ averaged over the next state using the following code: 

In [2]:
RC = { #CONDITIONAL REWARDS: conditional reward is 0 when there is no given reward, the probability 
      ('TL', 'R'): [0,-1,-2,0],
      ('TL', 'D'): [0, -1, -2, 0],
      ('TR', 'L'): [-1.5, 0, 0, 10],
      ('TR', 'D'): [-1, 0, 0, 15],
      ('BL', 'R'): [-2.5, 0, 0, 20],
      ('BL', 'U'): [-0.5, 0, 0, 5]
}

R = {}

# Calculate dot products for each state-action pair
for state_action in P.keys():
    prob_vector = P[state_action]
    reward_vector = RC[state_action]  # Default to [0,0,0,0] if no reward
    dot_product = sum(p*r for p,r in zip(prob_vector,reward_vector)) 
    R[state_action] = dot_product

print(R)

{('TL', 'R'): -1.1, ('TL', 'D'): -1.9000000000000001, ('TR', 'L'): -0.3500000000000001, ('TR', 'D'): 11.8, ('BL', 'R'): 17.75, ('BL', 'U'): 0.6}


We summarise this in the following table:

| State ($s$) | Action ($a$)| Reward $R(s,a)$|
|-----------|------------|----------|
| TL        | R          | -1.1   |
| TL        | D          | -1.9   |
| TR        | L          | -0.35   |
| TR        | D          | 11.8   |
| BL        | R          | 17.5   |
| BL        | U          | 0.6   |

Now we have all the relevant inputs needed for our value iteration algorithm:

In [3]:
from VIpackage import value_iteration

results = value_iteration(S, A, P, R, gamma=0.9, termination=1000)
print(results) 

({'TL': 'D', 'TR': 'D', 'BL': 'R', 'BR': None}, {'TL': 14.863870234277877, 'TR': 14.475496642170018, 'BL': 19.08774832108501, 'BR': 0})


We use a discount factor of $\gamma=0.9$, and force the algorithm to terminate if it has not identified a solution after $1000$ iterations of the algorithm. The discount factor reflects how much we care about past rewards when implementing this algorithm. The optimal policy for this discount factor is to go down if you are in the top row of the grid and right if you are in state BL. As BR is a terminal node, BR has no associated policy as there are no actions one can take in state BR. The other dictionary is a list of expected rewards given you follow the policy, if initialised in the given state, again the expected reward when initialised in the terminal node is 0.

More details on the values of iteration algorithm for Markov Decision Processes can be found here: https://artint.info/2e/html2e/ArtInt2e.Ch9.S5.html